
# Official ARNet on PanCollection WorldView-3

## الهدف

تدريب وتقييم معمارية **ARNet / ARConv** الرسمية على PanCollection WorldView-3 ذات الثماني حزم.

المعمارية:

```text
PAN 1 + LMS 8
      ↓
Head convolution
      ↓
ARConv U-Net encoder–decoder
      ↓
8-band residual
      ↓
Prediction = LMS + residual
```

إعدادات Script المستودع الرسمي:

```text
Adam
L1 Loss
LR = 0.0006
Epochs = 600
Batch size = 16
StepLR(step_size=200, gamma=0.8)
```

الـScript الرسمي يترك `hw_range` كـ`LOWER UPPER` دون أرقام. نستخدم هنا `[1, 9]` كإعداد إعادة تنفيذ معلن، لأنه متوافق مع Kernels الفردية التي يختارها التنفيذ من 3 إلى 7.

الـNotebook تدعم الاستكمال التلقائي من آخر Checkpoint.


In [1]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("فعّل T4 GPU.")

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:

from pathlib import Path
import os
import sys
import shutil
import subprocess

%cd /content

REPO_DIR = Path("/content/ARConv")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/Xueyangwang-will/ARConv.git",
        str(REPO_DIR),
    ],
    check=True,
)

!pip install -q h5py opencv-python-headless scipy scikit-image pandas matplotlib tqdm huggingface_hub

sys.path.insert(0, str(REPO_DIR / "models"))

from models import ARNet

print("Official ARNet imported.")


/content
Official ARNet imported.


In [3]:

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

WV3_DIR = (
    PROJECT_DIR
    / "Public_Datasets"
    / "PanCollection_WV3"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "WV3_ARNet_Official"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUTPUT_DIR / "best_wv3_arnet.pth"
LAST_PATH = OUTPUT_DIR / "last_wv3_arnet.pth"
HISTORY_PATH = OUTPUT_DIR / "wv3_arnet_history.json"
METRICS_PATH = OUTPUT_DIR / "wv3_arnet_validation_metrics.json"

print("WV3:", WV3_DIR)
print("Output:", OUTPUT_DIR)


Mounted at /content/drive
WV3: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3
Output: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_ARNet_Official


In [4]:

import h5py

h5_files = sorted(WV3_DIR.rglob("*.h5")) if WV3_DIR.exists() else []

def find_candidates(files):
    train = [
        path for path in files
        if (
            "wv3" in path.name.lower()
            and "train" in path.name.lower()
            and "valid" not in path.name.lower()
            and "test" not in path.name.lower()
        )
    ]

    valid = [
        path for path in files
        if (
            "wv3" in path.name.lower()
            and (
                "valid" in path.name.lower()
                or "validation" in path.name.lower()
            )
        )
    ]

    return train, valid

train_candidates, valid_candidates = find_candidates(h5_files)

if not train_candidates or not valid_candidates:
    from huggingface_hub import HfApi, snapshot_download

    WV3_DIR.mkdir(parents=True, exist_ok=True)

    repo_id = "elsting/PanCollection"
    api = HfApi()

    repo_files = api.list_repo_files(
        repo_id=repo_id,
        repo_type="dataset",
    )

    selected_files = [
        name for name in repo_files
        if (
            "wv3" in name.lower()
            and "training_data/" in name.lower()
            and name.lower().endswith(".h5")
        )
    ]

    for name in selected_files:
        print("Downloading:", name)

    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=selected_files,
        local_dir=str(WV3_DIR),
    )

    h5_files = sorted(WV3_DIR.rglob("*.h5"))
    train_candidates, valid_candidates = find_candidates(h5_files)

if not train_candidates:
    raise RuntimeError("WV3 Train H5 not found.")

if not valid_candidates:
    raise RuntimeError("WV3 Validation H5 not found.")

TRAIN_H5 = train_candidates[0]
VALID_H5 = valid_candidates[0]

print("Train:", TRAIN_H5)
print("Valid:", VALID_H5)

for path in [TRAIN_H5, VALID_H5]:
    with h5py.File(path, "r") as file:
        print("\n", path.name)

        for key in ["gt", "lms", "ms", "pan"]:
            if key in file:
                print(key, file[key].shape, file[key].dtype)


Train: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3/training_data/train_wv3_9714.h5
Valid: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/PanCollection_WV3/training_data/valid_wv3_9714.h5

 train_wv3_9714.h5
gt (9714, 8, 64, 64) float64
lms (9714, 8, 64, 64) float64
ms (9714, 8, 16, 16) float64
pan (9714, 1, 64, 64) float64

 valid_wv3_9714.h5
gt (1080, 8, 64, 64) float64
lms (1080, 8, 64, 64) float64
ms (1080, 8, 16, 16) float64
pan (1080, 1, 64, 64) float64


## Profile التدريب

In [5]:

import json
import math
import random
import numpy as np

TRAIN_PROFILE = "smoke"
# "smoke" ثم "colab_practical" ثم "official"

SEED = 42
MAX_DN = 2047.0
HW_RANGE = [1, 9]

if TRAIN_PROFILE == "smoke":
    TARGET_EPOCHS = 2
    BATCH_SIZE = 2
    ACCUMULATION_STEPS = 1
    VALIDATE_EVERY = 1

elif TRAIN_PROFILE == "colab_practical":
    TARGET_EPOCHS = 120
    BATCH_SIZE = 4
    ACCUMULATION_STEPS = 4
    VALIDATE_EVERY = 5

elif TRAIN_PROFILE == "official":
    TARGET_EPOCHS = 600
    BATCH_SIZE = 4
    ACCUMULATION_STEPS = 4
    VALIDATE_EVERY = 10

else:
    raise ValueError(TRAIN_PROFILE)

INITIAL_LR = 6e-4
STEP_SIZE = 200
GAMMA = 0.8
SAVE_EVERY = 5
NUM_WORKERS = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Profile:", TRAIN_PROFILE)
print("Target epochs:", TARGET_EPOCHS)
print("Effective batch:", BATCH_SIZE * ACCUMULATION_STEPS)
print("HW range:", HW_RANGE)


Profile: smoke
Target epochs: 2
Effective batch: 2
HW range: [1, 9]


In [6]:

from torch.utils.data import Dataset, DataLoader


class WV3Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = Path(h5_path)
        self._file = None

        with h5py.File(self.h5_path, "r") as file:
            for key in ["gt", "lms", "pan"]:
                if key not in file:
                    raise KeyError(f"{key} missing.")

            self.length = file["gt"].shape[0]

            print(
                self.h5_path.name,
                "samples:",
                self.length,
                "GT:",
                file["gt"].shape,
                "LMS:",
                file["lms"].shape,
                "PAN:",
                file["pan"].shape,
            )

    def _open(self):
        if self._file is None:
            self._file = h5py.File(self.h5_path, "r")

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        self._open()

        gt = np.asarray(
            self._file["gt"][index],
            dtype=np.float32,
        ) / MAX_DN

        lms = np.asarray(
            self._file["lms"][index],
            dtype=np.float32,
        ) / MAX_DN

        pan = np.asarray(
            self._file["pan"][index],
            dtype=np.float32,
        ) / MAX_DN

        return {
            "gt": torch.from_numpy(np.clip(gt, 0, 1)).float(),
            "lms": torch.from_numpy(np.clip(lms, 0, 1)).float(),
            "pan": torch.from_numpy(np.clip(pan, 0, 1)).float(),
        }


train_dataset = WV3Dataset(TRAIN_H5)
valid_dataset = WV3Dataset(VALID_H5)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))


train_wv3_9714.h5 samples: 9714 GT: (9714, 8, 64, 64) LMS: (9714, 8, 64, 64) PAN: (9714, 1, 64, 64)
valid_wv3_9714.h5 samples: 1080 GT: (1080, 8, 64, 64) LMS: (1080, 8, 64, 64) PAN: (1080, 1, 64, 64)
Train batches: 4857
Valid batches: 540


In [7]:

import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

model = ARNet(
    pan_channels=1,
    lms_channels=8,
).to(device)

criterion = nn.L1Loss()

optimizer = Adam(
    model.parameters(),
    lr=INITIAL_LR,
    betas=(0.9, 0.999),
)

scheduler = StepLR(
    optimizer,
    step_size=STEP_SIZE,
    gamma=GAMMA,
)

sample = next(iter(train_loader))

with torch.inference_mode():
    test_output = model(
        sample["pan"][:1].to(device),
        sample["lms"][:1].to(device),
        epoch=1,
        hw_range=HW_RANGE,
    )

print("Output:", test_output.shape)
print("Finite:", torch.isfinite(test_output).all().item())
print("Stored parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Output: torch.Size([1, 8, 64, 64])
Finite: True
Stored parameters: 15,923,720


In [8]:

from skimage.metrics import structural_similarity


def psnr_values(prediction, target):
    mse = torch.mean(
        (prediction - target) ** 2,
        dim=(1, 2, 3),
    )

    return 10.0 * torch.log10(
        1.0 / torch.clamp(mse, min=1e-12)
    )


def sam_values(prediction, target):
    dot = torch.sum(prediction * target, dim=1)

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction, dim=1)
        * torch.linalg.vector_norm(target, dim=1),
        min=1e-8,
    )

    cosine = torch.clamp(
        dot / denominator,
        -1 + 1e-7,
        1 - 1e-7,
    )

    return (
        torch.acos(cosine)
        * 180.0
        / math.pi
    ).mean(dim=(1, 2))


def ergas_values(prediction, target, scale=4):
    rmse = torch.sqrt(
        torch.mean(
            (prediction - target) ** 2,
            dim=(2, 3),
        )
    )

    target_mean = torch.mean(
        target,
        dim=(2, 3),
    ).abs().clamp_min(1e-6)

    return (
        100.0
        / scale
        * torch.sqrt(
            torch.mean(
                (rmse / target_mean) ** 2,
                dim=1,
            )
        )
    )


def ssim_values(prediction, target):
    prediction_np = prediction.detach().float().cpu().numpy()
    target_np = target.detach().float().cpu().numpy()

    results = []

    for sample_index in range(prediction_np.shape[0]):
        band_scores = []

        for band_index in range(prediction_np.shape[1]):
            band_scores.append(
                structural_similarity(
                    target_np[sample_index, band_index],
                    prediction_np[sample_index, band_index],
                    data_range=1.0,
                )
            )

        results.append(float(np.mean(band_scores)))

    return results


In [9]:

from tqdm.auto import tqdm


@torch.inference_mode()
def evaluate(evaluation_epoch):
    model.eval()

    metrics = {
        "l1": [],
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
    }

    for batch in tqdm(
        valid_loader,
        desc="WV3 validation",
        leave=False,
    ):
        gt = batch["gt"].to(device)
        lms = batch["lms"].to(device)
        pan = batch["pan"].to(device)

        prediction = model(
            pan,
            lms,
            epoch=evaluation_epoch,
            hw_range=HW_RANGE,
        ).clamp(0, 1)

        metrics["l1"].append(
            F.l1_loss(prediction, gt).item()
        )

        metrics["psnr"].extend(
            psnr_values(prediction, gt).cpu().tolist()
        )

        metrics["sam"].extend(
            sam_values(prediction, gt).cpu().tolist()
        )

        metrics["ergas"].extend(
            ergas_values(prediction, gt).cpu().tolist()
        )

        metrics["ssim"].extend(
            ssim_values(prediction, gt)
        )

    return {
        name: float(np.mean(values))
        for name, values in metrics.items()
    }


In [10]:

start_epoch = 1
best_psnr = -float("inf")
history = []

if LAST_PATH.exists():
    checkpoint = torch.load(
        LAST_PATH,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_psnr = checkpoint.get("best_psnr", best_psnr)
    history = checkpoint.get("history", [])

    print("Resume epoch:", start_epoch)


def save_checkpoint(path, epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "best_psnr": best_psnr,
            "history": history,
            "profile": TRAIN_PROFILE,
            "hw_range": HW_RANGE,
            "normalization": "DN / 2047",
        },
        path,
    )


print("Training:", start_epoch, "→", TARGET_EPOCHS)


Training: 1 → 2


In [11]:

for epoch in range(start_epoch, TARGET_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    losses = []

    progress = tqdm(
        train_loader,
        desc=f"ARNet {epoch}/{TARGET_EPOCHS}",
        leave=False,
    )

    for batch_index, batch in enumerate(progress, start=1):
        gt = batch["gt"].to(device, non_blocking=True)
        lms = batch["lms"].to(device, non_blocking=True)
        pan = batch["pan"].to(device, non_blocking=True)

        prediction = model(
            pan,
            lms,
            epoch=epoch,
            hw_range=HW_RANGE,
        )

        loss = criterion(prediction, gt)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch}."
            )

        (loss / ACCUMULATION_STEPS).backward()

        should_step = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(train_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())

        progress.set_postfix(
            l1=f"{loss.item():.5f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
        )

    scheduler.step()

    train_l1 = float(np.mean(losses))
    validation = None

    if (
        epoch == 1
        or epoch % VALIDATE_EVERY == 0
        or epoch == TARGET_EPOCHS
    ):
        validation = evaluate(epoch)

    history.append(
        {
            "epoch": epoch,
            "train_l1": train_l1,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "validation": validation,
        }
    )

    print(
        f"Epoch {epoch:03d} | Train L1 {train_l1:.6f}",
        end="",
    )

    if validation is not None:
        print(
            f" | PSNR {validation['psnr']:.3f}"
            f" | SSIM {validation['ssim']:.4f}"
            f" | SAM {validation['sam']:.3f}°"
            f" | ERGAS {validation['ergas']:.3f}"
        )

        if validation["psnr"] > best_psnr:
            best_psnr = validation["psnr"]
            save_checkpoint(BEST_PATH, epoch)
            print("Saved new best.")
    else:
        print()

    if epoch % SAVE_EVERY == 0 or epoch == TARGET_EPOCHS:
        save_checkpoint(LAST_PATH, epoch)

    with open(HISTORY_PATH, "w", encoding="utf-8") as file:
        json.dump(
            history,
            file,
            indent=2,
            ensure_ascii=False,
        )

print("Completed.")
print("Best:", BEST_PATH)
print("Last:", LAST_PATH)


ARNet 1/2:   0%|          | 0/4857 [00:00<?, ?it/s]

WV3 validation:   0%|          | 0/540 [00:00<?, ?it/s]

Epoch 001 | Train L1 0.016399 | PSNR 34.016 | SSIM 0.9301 | SAM 4.636° | ERGAS 3.322
Saved new best.


ARNet 2/2:   0%|          | 0/4857 [00:00<?, ?it/s]

WV3 validation:   0%|          | 0/540 [00:00<?, ?it/s]

Epoch 002 | Train L1 0.012784 | PSNR 34.711 | SSIM 0.9389 | SAM 4.386° | ERGAS 3.047
Saved new best.
Completed.
Best: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_ARNet_Official/best_wv3_arnet.pth
Last: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_ARNet_Official/last_wv3_arnet.pth


In [12]:

best_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"],
    strict=True,
)

best_epoch = int(best_checkpoint["epoch"])

final_metrics = evaluate(
    evaluation_epoch=max(best_epoch, 101)
)

reserved_kernels = {}

for name, module in model.named_modules():
    if hasattr(module, "reserved_NXY"):
        reserved_kernels[name] = (
            module.reserved_NXY
            .detach()
            .cpu()
            .tolist()
        )

payload = {
    "model": "Official ARNet",
    "dataset": "PanCollection WV3 Validation",
    "checkpoint_epoch": best_epoch,
    "hw_range": HW_RANGE,
    "normalization": "DN / 2047",
    "reserved_kernels": reserved_kernels,
    "metrics": final_metrics,
}

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps(payload, indent=2, ensure_ascii=False))
print("Saved:", METRICS_PATH)


WV3 validation:   0%|          | 0/540 [00:00<?, ?it/s]

{
  "model": "Official ARNet",
  "dataset": "PanCollection WV3 Validation",
  "checkpoint_epoch": 2,
  "hw_range": [
    1,
    9
  ],
  "normalization": "DN / 2047",
  "reserved_kernels": {
    "rb1.conv1": [
      3,
      3
    ],
    "rb1.conv2": [
      3,
      3
    ],
    "rb2.conv1": [
      3,
      3
    ],
    "rb2.conv2": [
      3,
      3
    ],
    "rb3.conv1": [
      3,
      3
    ],
    "rb3.conv2": [
      3,
      3
    ],
    "rb4.conv1": [
      3,
      3
    ],
    "rb4.conv2": [
      3,
      3
    ],
    "rb5.conv1": [
      3,
      3
    ],
    "rb5.conv2": [
      3,
      3
    ]
  },
  "metrics": {
    "l1": 0.012299797513211768,
    "psnr": 34.803771283891464,
    "ssim": 0.9398406713839108,
    "sam": 4.347352526419693,
    "ergas": 3.019367946325629
  }
}
Saved: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_ARNet_Official/wv3_arnet_validation_metrics.json



# التشغيل

1. شغّل أولًا:
   ```python
   TRAIN_PROFILE = "smoke"
   ```
2. بعد نجاح Epochs الاختبار، غيّر إلى:
   ```python
   TRAIN_PROFILE = "colab_practical"
   ```
3. لإعادة تنفيذ 600 Epoch:
   ```python
   TRAIN_PROFILE = "official"
   ```

الاستكمال يتم تلقائيًا من:

```text
WV3_ARNet_Official/last_wv3_arnet.pth
```

بعد الحصول على Checkpoint مناسب، افتح Notebook 15 لنقل النموذج إلى بياناتك المحلية ذات الست حزم.
